# Setting up the analysis by building a dataset
The building blocks of our package are built with the helper package brain2behaviour. 
Specifcally they're built off of the dataset class. Once you set up a data class, then you can just run them into the end2end cpm script and get your results

In [ ]:
import os
import pandas as pd
once_random_seeds=([0,42,19,123,10,69,33,1,1234,9245])
data_dir='/Output path to save pickle files'

## Load the behavioural and demographic dataset
### Choose your confounds

In [ ]:
restr_beh=pd.read_csv('<path to restricted behaviour>',index_col='Subject')
norm_beh=pd.read_csv('<path to standard behaviour>',index_col='Subject')
### put them together to get a behavioural and demographic df
all_beh=pd.concat([norm_beh,restr_beh],axis=1)

#### choose your confound columns 
conf_cats=['Acquisition','Gender','Age_in_Yrs','Height',
 'Weight','FS_IntraCranial_Vol','FS_BrainSeg_Vol','BPSystolic',
 'BPDiastolic','SSAGA_Income','SSAGA_Educ']

In [4]:
full_Set=all_beh[conf_cats].dropna()
subjects=full_Set.index

In [5]:
conf_cats=['Acquisition',
 'Gender',
'Age_in_Yrs',
 # 'Height',
 # 'Weight',
 'FS_IntraCranial_Vol',
 'FS_BrainSeg_Vol',
 'BPSystolic',
 'BPDiastolic',
 'SSAGA_Income',
 'SSAGA_Educ']

full_Set=all_beh[conf_cats].dropna()
full_Set=full_Set.loc[subjects]

### Get subject list

In [ ]:
import pandas as pd
list(pd.read_csv('../data_csvs/schaefer400_sa_discovery.csv',index_col=0).index)

In [ ]:
subj_list_all=list(pd.read_csv('<path to downloaded csv -- surface area loads fastest>',index_col=0).index)
subj_list_all=[int() for i in subj_list_all]
complete_subjects=list(full_Set.loc[[i for i in subj_list_all if i in full_Set.index]].index)

## when making dataset from full HCP we drop these subjects as they don't have family info available for pemrutations 
# no_fam=[122418,168240,376247] 
# subjects2drop=[i for i in no_fam if i in complete_subjects]
# complete_subjects=[i for i in complete_subjects if i not in subjects2drop]

### Set up the total surface area confounds

We've included the total left and right hemispheres in the `total_hemi_surface_area.csv'` folder so you don't need to recalculate it. 
However, if you wished to this is how you would recreate it

1.  On each left and right hemisphere you  run `wb_command -surface-vertex-areas < in surface > < out metric file >
2. Get a list of the metric files
3. Load the glasser atlas and select all vertices not in medial wall
4. Sum them
5. Add them to the confounds dataframe

In [ ]:
giftis=!ls /well/margulies/projects/data/SurfaceAreaGiftis/*L*area*
gifti_subjs=[int(i.split('/')[-1].split('.')[0]) for i in giftis]

from surfdist.load import load_cifti_labels,load_gifti_labels
import numpy as np
Lverts=load_gifti_labels('brain2behaviour/atlases/glasser/Glasser_2016.32k.L.label.gii')
del Lverts['???']
Lcort_mask=np.hstack(Lverts.values())

Rverts=load_gifti_labels('brain2behaviour/atlases/glasser/Glasser_2016.32k.R.label.gii')
del Rverts['???']
Rcort_mask=np.hstack(Rverts.values())

import nibabel as nib
lareas={}
rareas={}
for subj in complete_subjects:
    saL=nib.load(f'/well/margulies/projects/data/SurfaceAreaGiftis/{subj}.L.area.func.gii').darrays[0].data
    lareas[subj]=np.sum(saL[Lcort_mask])

    saR=nib.load(f'/well/margulies/projects/data/SurfaceAreaGiftis/{subj}.R.area.func.gii').darrays[0].data
    rareas[subj]=np.sum(saR[Rcort_mask])

AreaMetrics=pd.concat([pd.DataFrame([lareas],index=['Larea']),pd.DataFrame([rareas],index=['Rarea'])]).T

Build a confounds dataframe

In [ ]:
all_confounds=full_Set.loc[complete_subjects]
family_data=pd.DataFrame(all_beh['Family_ID'].loc[complete_subjects])
all_confounds=pd.concat([all_confounds,AreaMetrics],axis=1)

Get the behaviour of interest on its own

In [ ]:
### get it from the behaviour dataset
reading_data=all_beh['ReadEng_Unadj'].loc[subj_list_all]
### or from a previous file you've extracted it frmo 
reading_data=pd.read_csv(f'<file to ORRT scores. Alternately just extract from behaviour as above>',index_col=0)


## Make datasets

In [8]:
import numpy as np
import pandas as pd
import seaborn as sns
from brain2behaviour.dataset import BrainBehaviorDataset

Examples are shown for the Schaefer 400. 
This can be looped through for different atlases and such. 

In [ ]:
dirname='<some output directory path>'
dir_name=os.path.dirname(i)

temp_features=f'{dir_name}/features_tmp'
os.makedirs(temp_features, exist_ok=True)
### instantiate our class

### Distance
out_file=f'{dir_name}/Distance_file.pkl'
DistvReading=BrainBehaviorDataset('<Distance: dataframe or path to data csv or parquet file>',reading_data,all_confounds,
                                  (10,10),filepath=out_file,subjectList=complete_subjects,cv_seeds=once_random_seeds)
DistvReading.gen_CV_FamilyFoldsImplicit(family_data,'Family_ID')
DistvReading.save(overwrite=True)

### FC
out_file=f'{dir_name}/FC_file.pkl'
FCReading=BrainBehaviorDataset('<FC: dataframe or path to data csv or parquet file>',reading_data,all_confounds,
                                  (10,10),filepath=out_file,subjectList=complete_subjects,cv_seeds=once_random_seeds)
FCReading.gen_CV_FamilyFoldsImplicit(family_data,'Family_ID')
FCReading.save(overwrite=True)

### Surface area

out_file=f'{dir_name}/SA_file.pkl'
SAvReading=BrainBehaviorDataset('<FC: dataframe or path to data csv or parquet file>',reading_data,all_confounds,
                                  (10,10),filepath=out_file,subjectList=complete_subjects,cv_seeds=once_random_seeds)
SAvReading.gen_CV_FamilyFoldsImplicit(family_data,'Family_ID')
SAvReading.save(overwrite=True)


# Running CPM in the discovery cohort

Now that we have datasets for each modality, we simply need to run the script `End2EndCPM_wPerms.py`


`End2EndCPM_wPerms.py` puts everything together. It takes a dataset (`.pkl`) and a permutation index file (subjects × N perms CSV) then for each permutation:

1. Shuffles the behavioural labels using that permutation's subject-index column and saves a new permuted dataset `.pkl`
2. Generates a fold-level feature extraction worklist — one command per cross-validation fold
3. Submits a SLURM array job running `select_features_batch-SES.py` across all folds in parallel
4. Chains a dependent `CollectFeaturesandPredict-SES.py` job that fires automatically once all fold jobs complete
5. Repeats steps 1–4 for every permutation column in the CSV


These scripts rely heavily on the built in functionality to the helper `brain2behaviour` package

## Calling it for our example datasets
python End2EndCPM_wPerms.py --dataset=< Distance_file.pkl > --perms=./data/permutations/pset_single_1000perms.csv --task ReadEng_Unadj

python End2EndCPM_wPerms.py --dataset=< FC_file.pkl > --perms=./data/permutations/pset_single_1000perms.csv --task ReadEng_Unadj

python End2EndCPM_wPerms.py --dataset=< SA_file.pkl > --perms=./data/permutations/pset_single_1000perms.csv --task ReadEng_Unadj


That's it. Do that and the cross-validation with permutations in the discovery dataset will be done

#### A note on building datasets for the Lifespan HCP Aging cohort

Building the dataset is the same. However constructing the behaviours and confounds is quite different. 

The easiest way to do so is to use a created dataset from above as a template and to simply update the fields

We show below how we accessed the data and set up harmonized column names so we could test generalization

In [ ]:
### handling the reading measures
orrt=pd.read_csv('<lifespance hcp path>/orrt01.txt',delimiter='\t').iloc[1:]
orrt=orrt[['src_subject_id','tbx_reading_score']]
orrt.rename(columns={'src_subject_id':'subject_id','tbx_reading_score':'ReadEng_Unadj'},inplace=True)
orrt['subject_id']=[ int(i.split('HCA')[1]) for i in orrt['subject_id']]
orrt['ReadEng_Unadj']=[float(i) for i in orrt['ReadEng_Unadj'].values]
orrt.set_index('subject_id',inplace=True)
orrt.rename_axis('',inplace=True)
orrt.dropna(inplace=True)

In [ ]:
### handling of age and blood pressure
vitals_file='<lifespance hcp path>/vitals01.txt'
vital_conf=pd.read_csv(vitals_file,delimiter='\t').iloc[1:][['src_subject_id','sex','interview_age','bp_stand',]].dropna()
vital_conf['src_subject_id']=[int(i.strip('HCA')) for i in vital_conf['src_subject_id']]
vital_conf['interview_age']=[np.round(int(i)/12) for i in vital_conf['interview_age'].values]
BPSystolic=[float(i.split('/')[0]) for i in vital_conf['bp_stand']]
vital_conf['BPSystolic']=BPSystolic
BPDiastolic=[float(i.split('/')[1]) for i in vital_conf['bp_stand']]
vital_conf['BPDiastolic']=BPDiastolic
vital_conf.set_index('src_subject_id',inplace=True)
vital_conf.rename_axis('',inplace=True)
vital_conf.rename(columns={'sex':'Gender','interview_age':'Age_in_Yrs'},inplace=True)
vital_conf.drop(columns='bp_stand',inplace=True)

In [ ]:
### handling of education and income
ses_raw=pd.read_csv('<lifespance hcp path>/ssaga_cover_demo01.txt',delimiter='\t').iloc[1:,:]
ses_raw.index=[int(i.strip('HCA')) for i in ses_raw['src_subject_id']]
ses=ses_raw[['annual_fam_inc','bkgrnd_education']]
ses=ses.copy()
ses=ses.dropna()

ses['SSAGA_Educ'] = ses.loc[:,'bkgrnd_education'].map(edu_cats) ### harmonizes with hcp YA
ses=map_income_codes(ses,'annual_fam_inc','SSAGA_Income')
ses.drop(columns=['annual_fam_inc','bkgrnd_education'],inplace=True)
ses_confounds=ses
confounds=pd.concat([confounds,ses_confounds],join='inner',axis=1)
ses_subjects=list(confounds_w_ses.index)

We also provide the total left and right hemisphere surface areas 

see: ./data/validation_confound_total_surface_area.csv

### The lifspan HCP aging cohort distributes the raw freesurfer aseg files. 
The following function was used to parse for intracranial and brainseg volumes

In [ ]:
def parse_aseg_stats(path):
    """
    Parse a FreeSurfer aseg.stats file into two DataFrames.

    Returns
    -------
    measures_df : pd.DataFrame
        Rows are Measures from header lines like:
        '# Measure BrainSegVol, BrainSegVol, 123456, mm^3'
        Index = 'short' (e.g., 'BrainSegVol'), columns: ['name','short','value','units'].
    table_df : pd.DataFrame
        Main stats table (Index, SegId, NVoxels, Volume_mm3, StructName, ...).
        Numeric-looking columns are cast to numbers where possible.
    """
    measures = []
    table_rows = []
    col_headers = None

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            # Header lines
            if line.startswith("#"):
                # Parse Measure lines
                if line.startswith("# Measure"):
                    # Example: '# Measure BrainSegVol, BrainSegVol, 123456, mm^3'
                    m = re.match(
                        r"#\s*Measure\s+([^,]+)\s*,\s*([^,]+)\s*,\s*([^,]+)\s*,\s*([^,]+)",
                        line
                    )
                    if m:
                        name, short, value, units = [g.strip() for g in m.groups()]
                        try:
                            value = float(value)
                        except ValueError:
                            pass
                        measures.append(
                            {"name": name, "short": short, "value": value, "units": units}
                        )

                # Parse ColHeaders line for the table
                elif line.startswith("# ColHeaders"):
                    # Everything after '# ColHeaders' split on whitespace
                    parts = line.split()
                    # parts[0]='#', parts[1]='ColHeaders', remainder are headers
                    col_headers = parts[2:]

                # ignore other header lines
                continue

            # Data lines (table)
            parts = line.split()
            if col_headers is None:
                # Fallback if ColHeaders not present (rare)
                default = [
                    "Index", "SegId", "NVoxels", "Volume_mm3",
                    "StructName", "Mean", "StdDev", "Min", "Max", "Range"
                ]
                col_headers = default[:len(parts)]

            # If counts don't match, merge extras into the last column (defensive)
            if len(parts) != len(col_headers):
                if len(parts) > len(col_headers):
                    head = parts[:len(col_headers)-1]
                    tail = [" ".join(parts[len(col_headers)-1:])]
                    parts = head + tail
                else:
                    parts = parts + [None] * (len(col_headers) - len(parts))

            table_rows.append(dict(zip(col_headers, parts)))

    # Build DataFrames
    measures_df = pd.DataFrame(measures)
    if not measures_df.empty and "short" in measures_df.columns:
        measures_df = measures_df.set_index("short")

    table_df = pd.DataFrame(table_rows)

    # Attempt numeric casting where possible
    for c in table_df.columns:
        table_df[c] = pd.to_numeric(table_df[c], errors="ignore")

    return measures_df, table_df

With this in place, its a matter of simply reconstructing a confounds dataframe. 
Total surface areas can be calculated directly from the surfaces as done in the HCP YA discovery cohort.

In [ ]:

BrainBehaviorDataset(<'brain data df or csf'>,orrt,confounds,(10,10),filepath=<'Lifespan age outputs pkl'>,cv_seeds=once_random_seeds)
Distance.gen_CV_FoldsNaive(0.1,100)
Distance.feature_pthresh=0.01
Distance.save(overwrite=True)